# Historical Sentiment Scraper
Fetches and stores raw historical `NEWS_SENTIMENT` data from Alpha Vantage for every ticker in `config.TICKERS`. Paginates backward using a sliding `time_to` window and writes compressed `.jsonl.gz` files organised by calendar month under:

```
data/raw_sentiment/<YYYY>/<MM>/<TICKER>_<YYYYMMDD>_to_<YYYYMMDD>.jsonl.gz
```

## 1. Imports & Constants

In [1]:
import gzip
import json
import logging
import os
import time
from calendar import monthrange
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import requests
from dotenv import load_dotenv

from config import TICKERS

In [2]:
AV_BASE_URL: str = "https://www.alphavantage.co/query"
YEARS_BACK: int = 10
ARTICLES_PER_REQUEST: int = 1000
RATE_LIMIT_SLEEP_SECS: int = 65
DATA_ROOT: Path = Path("data/raw_sentiment")
LOG_DIR: Path = Path("logs")

# Alpha Vantage publishes article timestamps as YYYYMMDDTHHMMSS
AV_PUBLISHED_FMT: str = "%Y%m%dT%H%M%S"
# time_to / time_from query parameters use YYYYMMDDTHHMM (no seconds)
AV_PARAM_FMT: str = "%Y%m%dT%H%M"

## 2. Logging Setup

In [3]:
def setup_logging() -> None:
    """Configure root logger with a console handler and a persistent file handler."""
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(LOG_DIR / "scraper.log", mode="a", encoding="utf-8"),
        ],
    )

setup_logging()

## 3. Credentials & Date Window

In [4]:
load_dotenv()
API_KEY: Optional[str] = os.getenv("ALPHA_VANTAGE_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "ALPHA_VANTAGE_API_KEY not found in environment. "
        "Ensure it is defined in your .env file."
    )

# Upper bound for this backfill run: end of April 2024 (the month before the
# earliest existing raw_sentiment data, which starts in 2024/05). This
# prevents re-fetching articles that are already on disk.
TODAY: datetime = datetime(2024, 4, 30, 23, 59, 59)

# Lower bound: YEARS_BACK years before the current calendar date, not before TODAY.
_now: datetime = datetime.utcnow()
CUTOFF: datetime = datetime(_now.year - YEARS_BACK, _now.month, _now.day, 0, 0, 0)

logging.info("API key loaded successfully.")
logging.info("Scrape window: %s  →  %s", CUTOFF.date(), TODAY.date())
logging.info("Output root  : %s", DATA_ROOT.resolve())

2026-05-09 12:51:06 | INFO     | API key loaded successfully.
2026-05-09 12:51:06 | INFO     | Scrape window: 2016-05-09  →  2024-04-30
2026-05-09 12:51:06 | INFO     | Output root  : C:\Users\mattr\OneDrive\Desktop\src\SentimentAnalysisBot\data\raw_sentiment


## 4. Path Helper

In [5]:
def monthly_gz_path(ticker: str, year: int, month: int) -> Path:
    """
    Build the output path for a ticker's monthly .jsonl.gz file.

    Args:
        ticker: Stock symbol (e.g. 'NVDA').
        year:   Calendar year of the data.
        month:  Calendar month of the data (1–12).

    Returns:
        Path object with parent directories already created.

    Example:
        data/raw_sentiment/2024/05/NVDA_20240501_to_20240531.jsonl.gz
    """
    last_day = monthrange(year, month)[1]
    fname = f"{ticker}_{year}{month:02d}01_to_{year}{month:02d}{last_day:02d}.jsonl.gz"
    out_path = DATA_ROOT / str(year) / f"{month:02d}" / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    return out_path

## 5. Disk I/O — Append Articles to Compressed JSONL

In [6]:
def append_articles_to_disk(articles: list[dict], ticker: str) -> int:
    """
    Route each article to its calendar-month .jsonl.gz file and append it.

    Each line in the output file is a single JSON object (JSONL format).
    Python's gzip module supports multi-member gzip streams, so appending
    with mode 'ab' is fully readable without post-processing.

    Args:
        articles: Raw article dicts from the API 'feed' array.
        ticker:   Stock symbol used to build the output filename.

    Returns:
        Number of articles successfully written to disk.
    """
    monthly_buckets: dict[tuple[int, int], list[dict]] = {}
    for article in articles:
        try:
            pub_dt = datetime.strptime(article["time_published"], AV_PUBLISHED_FMT)
        except (KeyError, ValueError) as exc:
            logging.warning("Skipping article with unparseable timestamp: %s", exc)
            continue
        key = (pub_dt.year, pub_dt.month)
        monthly_buckets.setdefault(key, []).append(article)

    written = 0
    for (year, month), bucket in monthly_buckets.items():
        gz_path = monthly_gz_path(ticker, year, month)
        try:
            with gzip.open(gz_path, "ab") as gz_file:
                for article in bucket:
                    line = json.dumps(article, ensure_ascii=False) + "\n"
                    gz_file.write(line.encode("utf-8"))
                    written += 1
        except OSError as exc:
            logging.error("Disk write failed for %s: %s", gz_path, exc)

    return written

## 6. API Layer — Fetch a Single Page (with Rate-Limit Retry)

In [7]:
def fetch_page(
    session: requests.Session,
    ticker: str,
    api_key: str,
    time_to_dt: datetime,
    cutoff_dt: datetime,
) -> Optional[dict]:
    """
    Fetch a single NEWS_SENTIMENT page, retrying indefinitely on rate limits.

    Handles both hard rate limits (HTTP 429) and soft rate limits (HTTP 200
    with a JSON body containing 'Information': '... rate limit ...').
    Blocks with a sleep of RATE_LIMIT_SLEEP_SECS between retry attempts.

    Args:
        session:     Reusable requests.Session object.
        ticker:      Stock symbol to query.
        api_key:     Alpha Vantage API key.
        time_to_dt:  Fetch articles published strictly before this datetime.
        cutoff_dt:   Earliest date of interest; passed as time_from to the API.

    Returns:
        Parsed JSON dict on success, None on an unrecoverable error.
    """
    params: dict = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "time_from": cutoff_dt.strftime(AV_PARAM_FMT),
        "time_to": time_to_dt.strftime(AV_PARAM_FMT),
        "sort": "LATEST",
        "limit": ARTICLES_PER_REQUEST,
        "apikey": api_key,
    }

    while True:
        try:
            response = session.get(AV_BASE_URL, params=params, timeout=30)
        except requests.exceptions.ConnectionError as exc:
            logging.error("Connection error fetching %s | time_to=%s : %s", ticker, params["time_to"], exc)
            return None
        except requests.exceptions.Timeout:
            logging.error("Request timed out for %s | time_to=%s.", ticker, params["time_to"])
            return None
        except requests.exceptions.RequestException as exc:
            logging.error("Unexpected request error for %s: %s", ticker, exc)
            return None

        # --- Hard rate limit ---
        if response.status_code == 429:
            logging.warning(
                "HTTP 429 — rate limit hit for %s | time_to=%s. Sleeping %ds then retrying.",
                ticker, params["time_to"], RATE_LIMIT_SLEEP_SECS,
            )
            time.sleep(RATE_LIMIT_SLEEP_SECS)
            continue

        try:
            response.raise_for_status()
            data: dict = response.json()
        except requests.exceptions.HTTPError as exc:
            logging.error("HTTP error for %s: %s", ticker, exc)
            return None
        except json.JSONDecodeError as exc:
            logging.error("JSON decode error for %s: %s", ticker, exc)
            return None

        # --- Soft rate limit (200 OK with informational payload) ---
        info_msg: str = data.get("Information", "")
        if info_msg and "rate limit" in info_msg.lower():
            logging.warning(
                "Soft rate limit for %s | time_to=%s. Sleeping %ds then retrying. API message: %s",
                ticker, params["time_to"], RATE_LIMIT_SLEEP_SECS, info_msg,
            )
            time.sleep(RATE_LIMIT_SLEEP_SECS)
            continue

        return data

## 7. Ticker-Level Pagination Loop

In [8]:
def scrape_ticker(
    session: requests.Session,
    ticker: str,
    api_key: str,
    cutoff_dt: datetime,
    start_dt: datetime,
) -> None:
    """
    Paginate backward through all NEWS_SENTIMENT articles for a single ticker.

    After each successful page the time_to parameter is slid backward to
    (oldest_article_time – 1 minute). The loop terminates when:
      1. The oldest article in a response predates cutoff_dt, OR
      2. The feed returns fewer than ARTICLES_PER_REQUEST articles (exhausted), OR
      3. The API returns an unrecoverable error.

    Args:
        session:    Reusable requests.Session.
        ticker:     Stock symbol to scrape.
        api_key:    Alpha Vantage API key.
        cutoff_dt:  Earliest timestamp to persist (today minus YEARS_BACK years).
        start_dt:   Initial value for the time_to sliding parameter (today).
    """
    logger = logging.getLogger(__name__)
    time_to_dt: datetime = start_dt
    total_written: int = 0
    page_num: int = 0

    logger.info("── Starting  %-6s | window: %s → %s ──", ticker, cutoff_dt.date(), start_dt.date())

    while time_to_dt > cutoff_dt:
        page_num += 1
        t0 = time.monotonic()

        logger.info("%-6s | page %4d | time_to=%s", ticker, page_num, time_to_dt.strftime(AV_PARAM_FMT))

        data = fetch_page(session, ticker, api_key, time_to_dt, cutoff_dt)
        if data is None:
            logger.error("%-6s | Unrecoverable fetch error on page %d. Aborting ticker.", ticker, page_num)
            break

        articles: list[dict] = data.get("feed", [])
        if not articles:
            logger.info("%-6s | Empty feed returned. All available history exhausted.", ticker)
            break

        oldest_dt: Optional[datetime] = None
        in_window: list[dict] = []

        for article in articles:
            try:
                pub_dt = datetime.strptime(article["time_published"], AV_PUBLISHED_FMT)
            except (KeyError, ValueError):
                logging.debug("Skipping article with missing/malformed time_published.")
                continue

            if oldest_dt is None or pub_dt < oldest_dt:
                oldest_dt = pub_dt

            if pub_dt >= cutoff_dt:
                in_window.append(article)

        written = append_articles_to_disk(in_window, ticker)
        total_written += written
        elapsed = time.monotonic() - t0

        logger.info(
            "%-6s | page %4d | fetched=%4d | in_window=%4d | written=%4d | "
            "oldest=%s | total_written=%6d | elapsed=%.1fs",
            ticker, page_num,
            len(articles), len(in_window), written,
            oldest_dt.strftime(AV_PARAM_FMT) if oldest_dt else "N/A",
            total_written, elapsed,
        )

        # --- Termination checks ---
        if oldest_dt is None or oldest_dt <= cutoff_dt:
            logger.info(
                "%-6s | Oldest article (%s) is at or before cutoff (%s). Done.",
                ticker,
                oldest_dt.strftime(AV_PARAM_FMT) if oldest_dt else "N/A",
                cutoff_dt.strftime(AV_PARAM_FMT),
            )
            break

        if len(articles) < ARTICLES_PER_REQUEST:
            logger.info(
                "%-6s | Received %d < %d articles. Feed exhausted for this window.",
                ticker, len(articles), ARTICLES_PER_REQUEST,
            )
            break

        # Slide the window one minute before the oldest article we saw
        time_to_dt = oldest_dt - timedelta(minutes=1)

    logger.info("── Finished  %-6s | total articles written: %d ──", ticker, total_written)

## 8. Run the Scraper

In [9]:
logging.info("=" * 70)
logging.info("Historical Sentiment Scraper — started (UTC)")
logging.info("Tickers : %d", len(TICKERS))
logging.info("Window  : %s  →  %s", CUTOFF.date(), TODAY.date())
logging.info("Output  : %s", DATA_ROOT.resolve())
logging.info("=" * 70)

with requests.Session() as session:
    for idx, ticker in enumerate(TICKERS, start=1):
        logging.info("[%d/%d] ── Ticker: %s", idx, len(TICKERS), ticker)
        try:
            scrape_ticker(session, ticker, API_KEY, CUTOFF, TODAY)
        except Exception as exc:
            logging.exception(
                "Unexpected error while scraping %s — skipping. Error: %s",
                ticker, exc,
            )

logging.info("=" * 70)
logging.info("All tickers processed. Scraping complete.")
logging.info("=" * 70)

2026-05-09 12:51:06 | INFO     | ======================================================================
2026-05-09 12:51:06 | INFO     | Historical Sentiment Scraper — started (UTC)
2026-05-09 12:51:06 | INFO     | Tickers : 50
2026-05-09 12:51:06 | INFO     | Window  : 2016-05-09  →  2024-04-30
2026-05-09 12:51:06 | INFO     | Output  : C:\Users\mattr\OneDrive\Desktop\src\SentimentAnalysisBot\data\raw_sentiment
2026-05-09 12:51:06 | INFO     | ======================================================================
2026-05-09 12:51:06 | INFO     | [1/50] ── Ticker: NVDA
2026-05-09 12:51:06 | INFO     | ── Starting  NVDA   | window: 2016-05-09 → 2024-04-30 ──
2026-05-09 12:51:06 | INFO     | NVDA   | page    1 | time_to=20240430T2359
2026-05-09 12:51:09 | INFO     | NVDA   | page    1 | fetched=1000 | in_window=1000 | written=1000 | oldest=20220812T0850 | total_written=  1000 | elapsed=3.4s
2026-05-09 12:51:09 | INFO     | NVDA   | page    2 | time_to=20220812T0849
2026-05-09 12:51:12 | 